# Image only Regression Model
This code implements image only regression models using Resnet backbone architectures. The model extracts visual features from input images and predicts land cover composition percentages through a regression head. This serves as a baseline to evaluate how well visual information alone can capture area distribution without any additional semantic input.

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
DATA_ROOT = "/content/drive/MyDrive/subset"

In [6]:
import os

print(os.listdir(DATA_ROOT))

['captions_subset.csv', '.DS_Store', 'images']


In [7]:
DATA_ROOT = "/content/drive/MyDrive/subset"
CSV_PATH = os.path.join(DATA_ROOT, "captions_subset.csv")
IMAGE_DIR = os.path.join(DATA_ROOT, "images")

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 8
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [8]:
df = pd.read_csv(CSV_PATH)

print(df.head())
print("Dataset size:", len(df))

     filename  split  Tree  Shrub  Grass  Crop  Built-up  Barren  Water  \
0  267687.png  synth    85     13      2     0         0       0      0   
1  222448.png  synth    18      0     81     0         0       1      0   
2   14328.png  synth     2      0     65     1        22      10      0   
3  224249.png  synth    61      0     39     0         0       0      0   
4  218094.png  synth     0      0      5    93         2       0      0   

                                    hybrid_gemma3-4b  \
0  The image depicts a heavily forested mountaino...   
1  The image depicts a landscape dominated by ext...   
2  The image depicts a landscape dominated by ext...   
3  The image depicts a mountainous region dominat...   
4  The image depicts a landscape dominated by ext...   

                                  hybrid_qwen3-vl-8b  \
0  The scene is dominated by dense tree cover (85...   
1  The scene depicts a predominantly grass-covere...   
2  The scene is dominated by grasslands (65%

In [9]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Val:", len(val_df))

Train: 400
Val: 100


In [10]:
class AreaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row[TARGET_COLS].values.astype(np.float32))

        return image, target

In [11]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = AreaDataset(train_df, IMAGE_DIR, train_transform)
val_ds   = AreaDataset(val_df, IMAGE_DIR, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
class ImageRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, 7)

    def forward(self, x):
        logits = self.backbone(x)
        preds = torch.softmax(logits, dim=1) * 100
        return preds

model = ImageRegressor().to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 165MB/s]


In [13]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [14]:
from tqdm import tqdm

def train_epoch():
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc="Training")

    for imgs, targets in loop:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader.dataset)

In [15]:
def evaluate():
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(DEVICE)

            preds = model(imgs).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred)**0.5

    per_class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return mae, rmse, per_class_mae, y_true, y_pred


In [16]:
for epoch in range(EPOCHS):
    loss = train_epoch()
    mae, rmse, _, _, _ = evaluate()

    print(f"Epoch {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

Training: 100%|██████████| 13/13 [01:21<00:00,  6.31s/it, loss=220]


Epoch 1
  Loss: 508.9684
  MAE:  8.4404
  RMSE: 13.8764


Training: 100%|██████████| 13/13 [00:01<00:00,  9.14it/s, loss=137]


Epoch 2
  Loss: 195.1193
  MAE:  6.0421
  RMSE: 11.0209


Training: 100%|██████████| 13/13 [00:01<00:00,  9.23it/s, loss=100]


...

Epoch 6
  Loss: 56.3656
  MAE:  5.3811
  RMSE: 9.3193


Training: 100%|██████████| 13/13 [00:01<00:00,  8.97it/s, loss=79.2]


Epoch 7
  Loss: 53.1178
  MAE:  5.0814
  RMSE: 9.4202


Training: 100%|██████████| 13/13 [00:01<00:00,  8.95it/s, loss=33.9]


Epoch 8
  Loss: 42.3197
  MAE:  4.7988
  RMSE: 8.6086


In [17]:
mae, rmse, per_class_mae, y_true, y_pred = evaluate()

print("Final MAE:", mae)
print("Final RMSE:", rmse)

for col, score in zip(TARGET_COLS, per_class_mae):
    print(f"{col}: {score:.2f}")

Final MAE: 4.7988200187683105
Final RMSE: 8.608636786352177
Tree: 6.79
Shrub: 2.68
Grass: 10.81
Crop: 6.80
Built-up: 1.37
Barren: 3.16
Water: 1.98


# Image and Text Fusion Regression Model
This section extends the baseline by incorporating textual descriptions alongside images. Image features are combined with text embeddings obtained from a pretrained language model, and the fused representation is used for regression. This setup enables the model to leverage both visual and semantic information, allowing evaluation of the impact of multimodal learning on area estimation performance.

In [18]:
!pip -q install transformers

In [19]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [22]:
DATA_ROOT = "/content/drive/MyDrive/subset"
CSV_PATH = os.path.join(DATA_ROOT, "captions_subset.csv")
IMAGE_DIR = os.path.join(DATA_ROOT, "images")

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]
TEXT_COL = "vision_qwen3-vl-8b"

BATCH_SIZE = 16
IMG_SIZE = 224
EPOCHS = 8
LR = 1e-4
MAX_LEN = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [23]:
df = pd.read_csv(CSV_PATH)

print(df[[ "filename", TEXT_COL] + TARGET_COLS].head())
print("Dataset size:", len(df))

     filename                                 vision_qwen3-vl-8b  Tree  Shrub  \
0  267687.png  This satellite image shows a mountainous regio...    85     13   
1  222448.png  This satellite image shows a rugged, mountaino...    18      0   
2   14328.png  This satellite image shows a patchwork of agri...     2      0   
3  224249.png  This satellite image shows a rugged, mountaino...    61      0   
4  218094.png  This satellite image shows a patchwork of agri...     0      0   

   Grass  Crop  Built-up  Barren  Water  
0      2     0         0       0      0  
1     81     0         0       1      0  
2     65     1        22      10      0  
3     39     0         0       0      0  
4      5    93         2       0      0  
Dataset size: 500


In [24]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Val:", len(val_df))

Train: 400
Val: 100


In [25]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [26]:
class MultiModalAreaDataset(Dataset):
    def __init__(self, df, image_dir, text_col, target_cols, tokenizer, max_len=128, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.text_col = text_col
        self.target_cols = target_cols
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # image
        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # text
        text = str(row[self.text_col])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        # target
        target = torch.tensor(row[self.target_cols].values.astype(np.float32), dtype=torch.float32)

        return {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "target": target
        }

In [27]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = MultiModalAreaDataset(
    train_df, IMAGE_DIR, TEXT_COL, TARGET_COLS, tokenizer, max_len=MAX_LEN, transform=train_transform
)
val_ds = MultiModalAreaDataset(
    val_df, IMAGE_DIR, TEXT_COL, TARGET_COLS, tokenizer, max_len=MAX_LEN, transform=val_transform
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [28]:
class ImageTextRegressor(nn.Module):
    def __init__(self, num_outputs=7):
        super().__init__()

        # image encoder
        self.image_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        img_feat_dim = self.image_backbone.fc.in_features
        self.image_backbone.fc = nn.Identity()

        # text encoder
        self.text_backbone = AutoModel.from_pretrained("distilbert-base-uncased")
        text_feat_dim = self.text_backbone.config.hidden_size  # 768

        # projection layers
        self.image_proj = nn.Linear(img_feat_dim, 256)
        self.text_proj = nn.Linear(text_feat_dim, 256)

        # fusion head
        self.regressor = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image, input_ids, attention_mask):
        # image features
        img_feat = self.image_backbone(image)
        img_feat = self.image_proj(img_feat)

        # text features
        text_outputs = self.text_backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_feat = text_outputs.last_hidden_state[:, 0, :]
        text_feat = self.text_proj(text_feat)

        # fuse
        fused = torch.cat([img_feat, text_feat], dim=1)

        logits = self.regressor(fused)
        preds = torch.softmax(logits, dim=1) * 100.0
        return preds

In [29]:
model = ImageTextRegressor(num_outputs=len(TARGET_COLS)).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
def train_epoch():
    model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc="Training")

    for batch in loop:
        images = batch["image"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader.dataset)

In [31]:
def evaluate():
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            images = batch["image"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            targets = batch["target"]

            preds = model(images, input_ids, attention_mask).cpu().numpy()

            all_preds.append(preds)
            all_targets.append(targets.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    per_class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return mae, rmse, per_class_mae, y_true, y_pred

In [32]:
for epoch in range(EPOCHS):
    loss = train_epoch()
    mae, rmse, _, _, _ = evaluate()

    print(f"Epoch {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

Validation: 100%|██████████| 7/7 [00:00<00:00, 17.25it/s]


Epoch 1
  Loss: 412.3689
  MAE:  7.3090
  RMSE: 12.1068


Validation: 100%|██████████| 7/7 [00:00<00:00, 16.91it/s]


Epoch 2
  Loss: 152.6366
  MAE:  5.2780
  RMSE: 10.3100


Validation: 100%|██████████| 7/7 [00:00<00:00, 17.13it/s]


...

Epoch 6
  Loss: 78.0420
  MAE:  4.6874
  RMSE: 8.7336


Validation: 100%|██████████| 7/7 [00:00<00:00, 17.06it/s]


Epoch 7
  Loss: 75.5069
  MAE:  4.8442
  RMSE: 9.5089


Validation: 100%|██████████| 7/7 [00:00<00:00, 16.97it/s]

Epoch 8
  Loss: 75.6899
  MAE:  4.3998
  RMSE: 8.4014


In [33]:
mae, rmse, per_class_mae, y_true, y_pred = evaluate()

print("Final MAE:", mae)
print("Final RMSE:", rmse)

for col, score in zip(TARGET_COLS, per_class_mae):
    print(f"{col}: {score:.2f}")

Validation: 100%|██████████| 7/7 [00:00<00:00, 16.45it/s]

Final MAE: 4.399808406829834
Final RMSE: 8.401437509240886
Tree: 5.91
Shrub: 1.36
Grass: 10.85
Crop: 6.60
Built-up: 1.32
Barren: 3.24
Water: 1.52


The ResNet based models show a clear improvement when textual information is incorporated. The image only model achieves a validation MAE of 4.80 and RMSE of 8.61, while the multimodal model improves to 4.40 MAE and 8.40 RMSE, indicating that text contributes to more accurate area estimation.

At the class level, most categories benefit from multimodal fusion. Tree (MAE 6.79 to 5.91) and Shrub (MAE 2.68 to 1.36) show noticeable improvements, suggesting that textual descriptions help capture vegetation related features more effectively. Built up (MAE 1.37 to 1.32) and Water (MAE 1.98 to 1.52) also improve slightly.

Overall, the results show that ResNet benefits more from text compared to stronger transformer-based models, supporting the idea that simpler visual backbones rely more on additional textual cues under data-constrained settings.